In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(
            to right,
            limegreen,
            deepskyblue,
            limegreen
        );
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-06. **Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication**
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# MCP: Hosted Weather and Geocoding Servers
* **Hosted MCP server** 
    * Hosted in the cloud
    * Connect via a URL
    * No subprocess, no local install
* **`MCPServerStreamableHttp`** 
    * Connects to a remote MCP endpoint over HTTPS
    * Identical `async with` lifecycle as local `MCPServerStdio` 
* **Single server, multiple capabilities, or multiple servers**
    * One hosted server can bundle geocoding and weather tools together
    * No `@function_tool` needed
* **`cache_tools_list=True`** 
    * Fetches tool list once and reuses it across all turns
---

## Imports

In [ ]:
import os
from agents import Agent, trace
from agents.mcp import MCPServerStreamableHttp
from source.agent_loop import run_conversation

## MCP Server URL
* Server: **`https://weather.chukai.io/mcp`**
    * Open-Meteo weather APIs
    * Offers worldwide coverage, built-in geocoding, no API key

In [ ]:
WEATHER_MCP_URL = 'https://weather.chukai.io/mcp'

## Inspect the Available Tools
* `MCPServerStreamableHttp` fetches tool schemas over HTTPS — no subprocess involved

In [ ]:
async with MCPServerStreamableHttp(
    params={'url': WEATHER_MCP_URL},
    cache_tools_list=True
) as server:
    tools = await server.list_tools()

print(f'{len(tools)} tools:\n')
for t in tools:
    props = t.inputSchema.get('properties', {})
    req   = set(t.inputSchema.get('required', []))
    print(f'  {t.name}')
    print(f'    {t.description.splitlines()[0]}')
    for param, schema in props.items():
        marker = ' (required)' if param in req else ' (optional)'
        desc   = schema.get('description', '')[:80]
        print(f'    param: {param}{marker} — {desc}')
    print()

## Build and Run the Weather Agent


In [ ]:
INSTRUCTIONS = """
You are a weather assistant with worldwide coverage.

The server provides both geocoding and weather tools. When the user gives a city or place
name, use the geocoding tool first to get coordinates, then pass them to the weather tools.

Always format your responses as Markdown:
- Present forecasts and observations as Markdown tables
- Use bold for key values (temperature, wind, alerts)
- Report temperatures in both °F and °C
- Mention active alerts when relevant
"""

async with MCPServerStreamableHttp(
    params={'url': WEATHER_MCP_URL},
    cache_tools_list=True
) as server:
    agent = Agent(
        name='WeatherAssistant',
        model='gpt-5.5',
        instructions=INSTRUCTIONS,
        mcp_servers=[server]
    )
    with trace('02-05-06-hosted-mcp-weather-primary'):
        await run_conversation(
            'Weather assistant ready. Ask about forecasts, conditions, or alerts anywhere in the world!',
            agent
        )


## Example Prompts

| Prompt |
|---|
| `What's the current weather in Tokyo?` |
| `Give me the 7-day forecast for Miami` |
| `Are there any weather alerts in Oklahoma City?` |
| `Compare current conditions in Seattle, Phoenix, and Chicago` |
| `What are local forecasters saying about Austin this week?` |
| `What do the NWS meteorologists say about Boston this week?` |
| `What's the hourly forecast for New York City today?` |

## Fallback — If `weather.chukai.io` Is Down
* Run this cell instead of the one above
* Uses two separate hosted servers:
    * `geocoder.chukai.io` for geocoding 
    * `nws.caseyjhand.com` for US weather data

In [ ]:
from agents.mcp import MCPServerStreamableHttp

NWS_MCP_URL = 'https://nws.caseyjhand.com/mcp' # US only
GEOCODER_MCP_URL = 'https://geocoder.chukai.io/mcp' # IBM/chuk-mcp-geocoder

INSTRUCTIONS = """
You are a United States weather assistant.

Tools available:
- geocode (from geocoder server): city or place name → lat/lon
- batch_geocode: geocode multiple cities in one call
- NWS weather tools: forecasts, observations, alerts (require lat/lon)

Always format your responses as Markdown:
- Present forecasts and observations as Markdown tables
- Use bold for key values (temperature, wind, alerts)
- Report temperatures in both °F and °C
- Mention active alerts when relevant
"""

# parenthesized context manager in the `with` statement enables 
# managing a comma-separated list of MCP servers in this example
async with ( 
    MCPServerStreamableHttp(
        params={'url': GEOCODER_MCP_URL}, cache_tools_list=True,
    ) as geo_server,
    MCPServerStreamableHttp(
        params={'url': NWS_MCP_URL}, cache_tools_list=True,
    ) as nws_server
):
    agent = Agent(
        name='WeatherAssistant',
        model='gpt-5.5',
        instructions=INSTRUCTIONS,
        mcp_servers=[geo_server, nws_server] # two MCP servers
    )
    with trace('02-05-06-hosted-mcp-weather-fallback'):
        await run_conversation(
            'Weather assistant ready (fallback mode — US locations only).',
            agent,
        )

---
## Hosted MCP with Authentication
* Production MCP servers typically require authentication
* Pass credentials via the `headers` key inside `params` dictionary
* https://developers.openai.com/apps-sdk/build/auth

---

## MCP Servers vs. `@function_tool`
* Both expose tools to an agent
* Choice between them is **where the tool code runs and what can reach it**
* **Trade-off**: in-process vs. out-of-process

    | | `@function_tool` | Local MCP (`MCPServerStdio`) | Hosted MCP (`MCPServerStreamableHttp`) |
    |---|---|---|---|
    | Tool code runs | In your Python process | Separate subprocess | Remote server in the cloud |
    | What can use it | This agent only | Any MCP-compatible client on this machine | Any MCP-compatible client anywhere |
    | Setup | Decorate a function | Code + launch a server | Just provide a URL |
    | Latency | Lowest (direct call) | Low (local pipe) | Higher (HTTPS round-trip) |

### When to choose each 
* **`@function_tool`** 
    * Tools are specific to a given agent/app
    * Don't need to share them with other agents or non-Python clients
* **Local MCP server**
    * Reuse the same tools across multiple agents
    * Share with other MCP-compatible clients (Claude Desktop, VS Code, etc.) **on the same machine**
* **Hosted MCP Server**
    * Want the tools accessible from anywhere without distributing code
    * Examples
        * Shared team database server
        * Public API wrapper
        * Any capability that benefits from being centrally deployed
---

© 2026 by Deitel & Associates, Inc. All Rights Reserved.